In [1]:
# Ensure project root on sys.path for `src` imports
import sys
from pathlib import Path

cwd = Path.cwd()
for base in [cwd, cwd.parent, cwd.parent.parent]:
    if (base / "src").exists():
        sys.path.insert(0, str(base))
        break


# Student Guide: Generation (RAG)

Goal: turn retrieved context into a helpful answer.

What you’ll learn:
- The RAG loop: Retrieve → Read → Respond.
- How a prompt template uses both the user query and context.
- Why grounding in retrieved text reduces hallucinations.

Baseline here:
- We return a formatted summary (no external LLM call) so the project runs offline.
- To use a real LLM, replace `generate_answer` with an API call but keep the prompt structure.

References:
- Retrieval‑Augmented Generation: https://arxiv.org/abs/2005.11401
- Prompt engineering concepts: https://platform.openai.com/docs/guides/prompt-engineering

Try this:
- Add citations by including the chunk indices/paths in the final answer.
- Experiment with temperature and style when you plug in a real LLM.


# 04 - Generation

Assemble a prompt with retrieved context and produce an answer (dummy offline generator by default).



In [2]:
from src.retrieve import retrieve
from src.generate import generate_answer

query = "Where can I see Macbeth in Colorado this summer?"
snippets = retrieve(query, k=5)
answer = generate_answer(query, snippets)
MAX_OUT = 2000
print(answer[:MAX_OUT] + ("..." if len(answer) > MAX_OUT else ""))


Based on the context, here are relevant details for your query: Where can I see Macbeth in Colorado this summer?

*** START OF THE PROJECT GUTENBERG EBOOK 100 *** The Complete Works of William Shakespeare by William Shakespeare Contents THE SONNETS ALL’S WELL THAT ENDS WELL THE TRAGEDY OF ANTONY AND CLEOPATRA AS YOU LIKE IT THE COMEDY OF ERRORS THE TRAGEDY OF CORIOLANUS CYMBELINE THE TRAGEDY OF HAMLET, PRINCE OF DENMARK THE FIRST PART OF KING HENRY THE FOURTH THE SECOND PART OF KING HENRY THE FOURTH THE LIFE OF KING HENRY THE FIFTH THE FIRST PART OF HENRY THE SIXTH THE SECOND PART OF KING HENRY THE SIXTH THE THIRD PART OF KING HENRY THE SIXTH KING HENRY THE EIGHTH THE LIFE AND DEATH OF KING JOHN THE TRAGEDY OF JULIUS CAESAR THE TRAGEDY OF KING LEAR LOVE’S LABOUR’S LOST THE TRAGEDY OF MACBETH MEASURE FOR MEASURE THE MERCHANT OF VENICE THE MERRY WIVES OF WINDSOR A MIDSUMMER NIGHT’S DREAM MUCH ADO ABOUT NOTHING THE TRAGEDY OF OTHELLO, THE MOOR OF VENICE PERICLES, PRINCE OF TYRE KING RICHA

# 04 – Generation

Assemble a simple answer from retrieved context (no external LLM required).


In [3]:
from pathlib import Path

from src.ingest import load_raw_documents
from src.embed import train_tfidf, embed_query
from src.retrieve import retrieve_top_k
from src.generate import simple_generate

processed_dir = "data/processed"
raw_dir = "data/raw"
if not any(Path(processed_dir).glob("*.txt")):
    documents = load_raw_documents(raw_dir)
else:
    documents = [p.read_text(encoding="utf-8") for p in Path(processed_dir).glob("*.txt")]

vectorizer, matrix = train_tfidf(documents)
q = "Where can I see Macbeth in Colorado this summer?"
qv = embed_query(q, vectorizer)
results = retrieve_top_k(qv, matrix, documents, top_k=5)
print(simple_generate(q, results))


Here are the most relevant context snippets I found.
- Hear it not, Duncan, for it is a knell That summons thee to heaven or to hell. [_Exit._] SCENE II. The same. Enter Lady Macbeth. LADY MACBETH. That which hath made them drunk hath made me bold: What hath quench’d them hath given me fire.—Hark!—Peace! It was the owl that shriek’d, the fatal bellman, Which gives the stern’st good night. He is about it. The doors are open; and the surfeited grooms Do mock their charge with snores: I have drugg’d their possets, That death and nature do contend about them, Whether they live or die. MACBETH. [_Within._] Who’s there?—what, ho! LADY MACBETH. Alack! I am afraid they have awak’d, And ’tis not done. Th’ attempt and not the deed Confounds us.—Hark!—I laid their daggers ready; He could not miss ’em.—Had he not resembled My father as he slept, I had done’t.—My husband! Enter Macbeth. MACBETH. I have done the deed.—Didst thou not hear a noise? LADY MACBETH. I heard the owl scream and the crickets